# 34 View-Consistency FedMoX-DQA Loop

Learning-first self-only loop. It keeps FedMoX-like FL settings where promising: 50 rounds, 33.3% client sampling, one local checkpoint per client, sparse MoE routing. The new DQA signal is identity+hflip stable pseudoGT, used for staged training and early stopping.

In [ ]:
from __future__ import annotations

import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "dynamic_quality_aware_classwise_aggregation").exists():
    REPO_ROOT = Path("/app/Object_Detection")

PROJECT_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation" / "scene_daynight_dqa"
RUNNER = PROJECT_ROOT / "aggressive_dqamox" / "scripts" / "run_34_view_consistency_fedmox_loop.py"
LOG_DIR = PROJECT_ROOT / "aggressive_dqamox" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f"34_view_consistency_fedmox_loop_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}.log"

cmd = [
    sys.executable,
    str(RUNNER),
    "--target-map50", "0.60",
    "--client-limit", "3000",
    "--client-sampling-ratio", "0.333",
    "--gpus", "2",
    "--batch-size", "80",
    "--workers", "8",
]

print(" ".join(cmd))
print("log:", LOG_PATH)
with LOG_PATH.open("w", encoding="utf-8") as log:
    proc = subprocess.run(cmd, cwd=REPO_ROOT, stdout=log, stderr=subprocess.STDOUT)
print("returncode:", proc.returncode)
print(LOG_PATH.read_text(encoding="utf-8", errors="replace")[-8000:])
if proc.returncode not in (0, 2):
    raise SystemExit(proc.returncode)


In [ ]:
from __future__ import annotations

import csv
from pathlib import Path

summary_path = PROJECT_ROOT / "aggressive_dqamox" / "reports" / "34_view_consistency_fedmox_loop_summary.csv"
rows = list(csv.DictReader(summary_path.open(encoding="utf-8"))) if summary_path.exists() else []
for row in rows[-10:]:
    print(
        row.get("trial"),
        row.get("stage"),
        row.get("status"),
        row.get("best_label"),
        row.get("best_map50"),
        row.get("best_map50_95"),
        row.get("gain_vs_warmup"),
    )
